# SmartGlove — Gesture Recognition Training
## BISINDO Recognition with BiLSTM + Attention
Core pipeline: Import → Training → JSON + TFLite Conversion

## 0. All Imports

In [ ]:
# Standard libraries
import os
import json
import warnings
from pathlib import Path
from datetime import datetime

# Data processing
import numpy as np
import pandas as pd

# Machine Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

print(f"TensorFlow: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")
print("✅ All imports ready\n")

## 1. Load Gesture List & Data

In [ ]:
def load_gesture_list(filename='bisindo_gesture_list.txt'):
    """Load gesture labels from file (CATEGORY,label format)"""
    gestures, categories = [], {}
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split(',', 1)
            if len(parts) == 2:
                cat, label = parts[0].strip(), parts[1].strip()
                idx = len(gestures)
                gestures.append(label)
                categories.setdefault(cat, []).append(idx)
    return gestures, categories

gestures, categories = load_gesture_list()
num_gestures = len(gestures)

print(f"Total gestures: {num_gestures}")
for cat, idxs in categories.items():
    print(f"  {cat}: {len(idxs)} gestures")

In [ ]:
def load_data_from_folders(base_path='datashet', gestures=None, categories=None):
    """Load CSV data from datashet/ subfolder structure"""
    X_raw, y = [], []
    label_to_idx = {g: i for i, g in enumerate(gestures)}
    
    for cat, cat_idxs in categories.items():
        cat_dir = os.path.join(base_path, cat.lower())
        if not os.path.exists(cat_dir):
            print(f"  ⚠️  {cat_dir} not found")
            continue
        
        cat_count = 0
        for csv_file in Path(cat_dir).glob('*.csv'):
            try:
                df = pd.read_csv(csv_file)
                drop_cols = [c for c in ['timestamp', 'repetition'] if c in df.columns]
                sensor_df = df.drop(columns=drop_cols)
                
                if sensor_df.shape[1] != 22:
                    continue
                
                data = sensor_df.values.astype(np.float32)
                if len(data) == 0:
                    continue
                
                # Parse label from filename (format: label_rep1_timestamp.csv)
                fname = csv_file.stem
                raw_label = fname[:fname.index('_rep')].replace('_', ' ') if '_rep' in fname else fname.replace('_', ' ')
                
                if raw_label in label_to_idx:
                    label_idx = label_to_idx[raw_label]
                else:
                    matched = [g for g in gestures if g in raw_label or raw_label in g]
                    if not matched:
                        continue
                    label_idx = label_to_idx[matched[0]]
                
                X_raw.append(data)
                y.append(label_idx)
                cat_count += 1
            except Exception as e:
                continue
        
        print(f"  {cat}: {cat_count} recordings")
    
    print(f"\nTotal: {len(X_raw)} recordings")
    return X_raw, y

X_raw, y_raw = load_data_from_folders('datashet', gestures, categories)

if len(X_raw) > 0:
    seq_lengths = [len(s) for s in X_raw]
    print(f"Sequence length: min={min(seq_lengths)}, max={max(seq_lengths)}, median={np.median(seq_lengths):.0f}")

## 2. Preprocessing & Configuration

In [ ]:
# Import preprocessing if available
try:
    from advanced_gesture_recognition import (
        GloveSensorPreprocessor,
        CATEGORY_WINDOW,
        NUM_TOTAL_FEATURES,
        SAMPLING_RATE,
        CONFIDENCE_THRESHOLD,
    )
    print("✅ Using advanced preprocessor")
except ImportError:
    # Fallback to defaults
    CATEGORY_WINDOW = {'ALL': 80}
    NUM_TOTAL_FEATURES = 22
    SAMPLING_RATE = 100
    CONFIDENCE_THRESHOLD = 0.72
    GloveSensorPreprocessor = None
    print("⚠️  Using default config")

WINDOW_SIZE = CATEGORY_WINDOW['ALL']
print(f"Window size: {WINDOW_SIZE}, Features: {NUM_TOTAL_FEATURES}, Threshold: {CONFIDENCE_THRESHOLD}")

In [ ]:
# Preprocess data
print(f"\nPreprocessing {len(X_raw)} sequences...")

if GloveSensorPreprocessor:
    preprocessor = GloveSensorPreprocessor()
    preprocessor.fit(X_raw)
    X_processed = preprocessor.batch_transform(X_raw, WINDOW_SIZE)
    scaler_dict = preprocessor.to_dict()
else:
    # Simple padding fallback
    def simple_pad(sequences, window_size=80):
        result = []
        for seq in sequences:
            if len(seq) < window_size:
                padded = np.pad(seq, ((0, window_size - len(seq)), (0, 0)), mode='constant')
            else:
                padded = seq[:window_size]
            result.append(padded)
        return np.array(result)
    
    X_processed = simple_pad(X_raw, WINDOW_SIZE)
    scaler_dict = {'mean': 0, 'scale': 1}  # Dummy scaler

y_processed = np.array(y_raw)
print(f"Processed shape: {X_processed.shape}")
print(f"Labels shape: {y_processed.shape}")

## 3. Train/Validation Split & Augmentation

In [ ]:
# Split
X_train, X_val, y_train, y_val = train_test_split(
    X_processed, y_processed,
    test_size=0.2, random_state=42, stratify=y_processed
)

print(f"Train: {X_train.shape} | Val: {X_val.shape}")

# Try augmentation
try:
    from sensor_augmentation import SensorDataAugmenter
    augmenter = SensorDataAugmenter(seed=42)
    X_train, y_train = augmenter.augment_balanced(X_train, y_train, target_per_class=45, verbose=False)
    print(f"After augmentation: {X_train.shape}")
except ImportError:
    print("Augmentation skipped")

## 4. Build Model

In [ ]:
# Try advanced model builder
try:
    from advanced_gesture_recognition import build_bilstm_attention_model
    model = build_bilstm_attention_model(
        num_gestures=num_gestures,
        window_size=WINDOW_SIZE,
        num_features=NUM_TOTAL_FEATURES,
        lstm_units=128,
        dense_units=128,
        dropout_rate=0.35,
    )
    print("✅ BiLSTM + Attention model built")
except ImportError:
    # Fallback LSTM
    print("⚠️  Building simple LSTM")
    model = keras.Sequential([
        layers.Input(shape=(WINDOW_SIZE, NUM_TOTAL_FEATURES)),
        layers.LSTM(128, return_sequences=True),
        layers.Dropout(0.3),
        layers.LSTM(64),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_gestures, activation='softmax')
    ])

model.summary()
print(f"\nParameters: {model.count_params():,}")

## 5. Compile & Train

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0005, clipnorm=1.0),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=20,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=7,
        min_lr=1e-7, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        'best_gesture_model.keras',
        monitor='val_accuracy', save_best_only=True, verbose=0
    ),
]

print(f"Training on {len(X_train)} samples for {num_gestures} gestures...\n")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=120,
    batch_size=32,
    callbacks=callbacks,
    verbose=1,
)
print("\n✅ Training complete!")

## 6. Evaluate

In [ ]:
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)

print(f"Train: {train_acc*100:.2f}% (loss: {train_loss:.4f})")
print(f"Val:   {val_acc*100:.2f}% (loss: {val_loss:.4f})")

# Per-gesture accuracy
y_pred_probs = model.predict(X_val, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

per_gesture_acc = {}
for gesture_idx in range(num_gestures):
    mask = y_val == gesture_idx
    if mask.sum() > 0:
        acc = (y_pred[mask] == gesture_idx).mean()
        per_gesture_acc[gestures[gesture_idx]] = acc

print(f"\nPer-gesture accuracy:")
print(f"  Mean:   {np.mean(list(per_gesture_acc.values()))*100:.1f}%")
print(f"  Median: {np.median(list(per_gesture_acc.values()))*100:.1f}%")

# Show classification report
present_labels = sorted(np.unique(np.concatenate([y_val, y_pred])))
target_names = [gestures[i] for i in present_labels]
print("\n" + classification_report(y_val, y_pred, labels=present_labels, target_names=target_names, digits=3))

## 7. Save Model to Keras Format

In [ ]:
model.save('best_gesture_model.keras')
print("✅ Saved: best_gesture_model.keras")

# Check file size
import os
size_mb = os.path.getsize('best_gesture_model.keras') / (1024 * 1024)
print(f"   Size: {size_mb:.1f} MB")

## 8. Save Metadata to JSON

In [ ]:
metadata = {
    'training_info': {
        'date': datetime.now().isoformat(),
        'train_accuracy': float(train_acc),
        'val_accuracy': float(val_acc),
        'num_epochs': len(history.history['loss']),
    },
    'model_config': {
        'num_gestures': int(num_gestures),
        'window_size': int(WINDOW_SIZE),
        'num_features': int(NUM_TOTAL_FEATURES),
        'sampling_rate': int(SAMPLING_RATE),
        'confidence_threshold': float(CONFIDENCE_THRESHOLD),
    },
    'preprocessing': scaler_dict if isinstance(scaler_dict, dict) else {'mean': 0, 'scale': 1},
    'gesture_labels': gestures,
    'categories': {cat: idxs for cat, idxs in categories.items()},
}

with open('model_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("✅ Saved: model_metadata.json")
print(f"   Gestures: {num_gestures}")
print(f"   Categories: {list(categories.keys())}")

## 9. Convert to TFLite

In [ ]:
print("Converting to TFLite...\n")

# Float32 TFLite
try:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,
        tf.lite.OpsSet.SELECT_TF_OPS
    ]
    converter.experimental_enable_resource_variables = False
    tflite_model = converter.convert()
    
    with open('gesture_model_f32.tflite', 'wb') as f:
        f.write(tflite_model)
    
    size_kb = len(tflite_model) / 1024
    print(f"✅ Saved: gesture_model_f32.tflite ({size_kb:.1f} KB)")
except Exception as e:
    print(f"❌ Float32 conversion failed: {e}")

# INT8 TFLite (quantized)
try:
    print("\nQuantizing to INT8...")
    converter_i8 = tf.lite.TFLiteConverter.from_keras_model(model)
    converter_i8.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,
        tf.lite.OpsSet.SELECT_TF_OPS
    ]
    converter_i8.optimizations = [tf.lite.Optimize.DEFAULT]
    converter_i8.experimental_enable_resource_variables = False
    
    # Representative dataset for quantization
    def rep_dataset():
        for i in range(min(100, len(X_train))):
            yield [X_train[i:i+1].astype(np.float32)]
    
    converter_i8.representative_dataset = rep_dataset
    tflite_i8 = converter_i8.convert()
    
    with open('gesture_model_int8.tflite', 'wb') as f:
        f.write(tflite_i8)
    
    size_kb_i8 = len(tflite_i8) / 1024
    compression = len(tflite_model) / len(tflite_i8)
    print(f"✅ Saved: gesture_model_int8.tflite ({size_kb_i8:.1f} KB)")
    print(f"   Compression: {compression:.1f}x smaller than Float32")
except Exception as e:
    print(f"⚠️  INT8 conversion skipped: {e}")

## 10. Summary & Files Generated

In [ ]:
print("\n" + "="*70)
print("✅ TRAINING COMPLETE - FILES GENERATED:")
print("="*70)

files_to_check = [
    ('best_gesture_model.keras', 'Keras native format (full precision)'),
    ('gesture_model_f32.tflite', 'TFLite Float32 (Android)'),
    ('gesture_model_int8.tflite', 'TFLite INT8 quantized (smaller)'),
    ('model_metadata.json', 'Configuration & gesture labels'),
]

for fname, description in files_to_check:
    if os.path.exists(fname):
        size = os.path.getsize(fname)
        if size > 1024*1024:
            size_str = f"{size/(1024*1024):.1f} MB"
        else:
            size_str = f"{size/1024:.1f} KB"
        print(f"✓ {fname:30s} {size_str:>10s} - {description}")
    else:
        print(f"✗ {fname:30s} {'NOT FOUND':>10s} - {description}")

print("\n" + "="*70)
print(f"Model accuracy: {val_acc*100:.1f}%")
print(f"Total gestures: {num_gestures}")
print(f"Ready for Android deployment!")
print("="*70)